# TrumpetJudge - Precompute All Embeddings 🎺💾

**Run this ONCE** to encode all audio files. Then train locally anytime!

## What this does:
1. **Augment ALL audio** - Creates augmented versions of each clip
2. **Encode everything** - Runs PANNs encoder on all audio (original + augmented)
3. **Download** - Get the embeddings file to use locally

## After this notebook:
- Add labels anytime in `data/labels/`
- Train instantly: `python ml/train_fast.py --embeddings ... --train_csv ... --val_csv ...`
- No re-encoding needed!

## Requirements
- GPU runtime (A100 recommended)
- Audio files in Google Drive


In [ ]:
# Cell 1: Install dependencies
%pip install -q panns-inference soundfile tqdm audiomentations


In [ ]:
# Cell 2: Verify GPU
import torch

print("=" * 50)
print("GPU Configuration")
print("=" * 50)

if torch.cuda.is_available():
    print(f"✅ CUDA available: True")
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ No GPU detected!")
    print("Go to Runtime → Change runtime type → A100 GPU")


In [ ]:
# Cell 3: Clone repo + mount Google Drive
import os
from google.colab import drive

# ============================================
# CONFIGURE: Path to your audio folder in Google Drive
# ============================================
DRIVE_AUDIO_PATH = "Audio/audio"  # Change if different
# ============================================

os.chdir("/content")

# Clone from GitHub
if not os.path.exists("/content/TrumpetJudge"):
    print("📦 Cloning from GitHub...")
    !git clone https://github.com/AdnanKapadia/TrumpetJudge.git
    print("✅ Cloned!")
else:
    print("✅ Repo already cloned, pulling latest...")
    os.chdir("/content/TrumpetJudge")
    !git pull

# Mount Google Drive
print("\n📁 Mounting Google Drive...")
drive.mount('/content/drive')

# Link audio folder
os.chdir("/content/TrumpetJudge")
drive_audio_full = f"/content/drive/MyDrive/{DRIVE_AUDIO_PATH}"

if os.path.exists(drive_audio_full):
    import subprocess
    subprocess.run(["rm", "-rf", "data/audio"], check=True)
    subprocess.run(["ln", "-s", drive_audio_full, "data/audio"], check=True)
    print(f"✅ Linked audio from Google Drive")
else:
    print(f"❌ Audio not found at: {drive_audio_full}")

# Verify
print(f"\n📂 Working directory: {os.getcwd()}")
audio_files = [f for f in os.listdir('data/audio') if f.endswith('.wav')] if os.path.exists('data/audio') else []
print(f"📁 Audio files found: {len(audio_files)}")


In [ ]:
# Cell 4: Check to_label.csv exists
import os
import pandas as pd

os.chdir("/content/TrumpetJudge")

if os.path.exists("data/to_label.csv"):
    df = pd.read_csv("data/to_label.csv")
    print(f"✅ Found to_label.csv with {len(df)} audio entries")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nSample entries:")
    print(df[['id', 'path']].head())
else:
    print("❌ data/to_label.csv not found!")
    print("Make sure your repo has this file.")


In [ ]:
# Cell 5: Precompute embeddings for ALL original audio
import os
os.chdir("/content/TrumpetJudge")

if os.path.exists("data/embeddings/all_audio.pt"):
    print("✅ Embeddings already exist! Skipping...")
    import torch
    data = torch.load("data/embeddings/all_audio.pt", weights_only=False)
    print(f"   {data['num_samples']} samples")
    print(f"   Has labels: {data.get('has_labels', 'Yes (legacy)')}")
else:
    print("🔄 Precomputing embeddings for ALL original audio...")
    print("   This takes ~10-20 minutes depending on audio count\n")
    
    !python ml/precompute.py \
        --csv data/to_label.csv \
        --output data/embeddings/all_audio.pt \
        --no_labels \
        --batch_size 16


## Optional: Augmented Embeddings

Run the next two cells if you want to pre-generate augmented audio embeddings too.
This creates more training data variety but takes longer and creates larger files.


In [ ]:
# Cell 6 (OPTIONAL): Create augmented audio files
import os
os.chdir("/content/TrumpetJudge")

# ============================================
# CONFIGURE: Number of augmentations per file
# ============================================
NUM_AUGMENTS = 5  # More = better variety, but larger files
# ============================================

if os.path.exists("data/all_augmented.csv"):
    print("✅ Augmented audio already exists! Skipping...")
    import pandas as pd
    aug_df = pd.read_csv("data/all_augmented.csv")
    print(f"   {len(aug_df)} augmented samples")
else:
    print(f"🔄 Creating {NUM_AUGMENTS} augmented versions of ALL audio...")
    print("   This takes ~15-30 minutes\n")
    
    !python ml/augment_offline.py \
        --input_csv data/to_label.csv \
        --output_dir data/audio_augmented_all \
        --output_csv data/all_augmented.csv \
        --num_augments {NUM_AUGMENTS}


In [ ]:
# Cell 7 (OPTIONAL): Precompute embeddings for augmented audio
import os
os.chdir("/content/TrumpetJudge")

if not os.path.exists("data/all_augmented.csv"):
    print("⚠️ Run the previous cell first to create augmented audio!")
elif os.path.exists("data/embeddings/all_augmented.pt"):
    print("✅ Augmented embeddings already exist! Skipping...")
    import torch
    data = torch.load("data/embeddings/all_augmented.pt", weights_only=False)
    print(f"   {data['num_samples']} samples")
else:
    print("🔄 Precomputing embeddings for augmented audio...")
    print("   This takes ~30-60 minutes (lots of files!)\n")
    
    !python ml/precompute.py \
        --csv data/all_augmented.csv \
        --output data/embeddings/all_augmented.pt \
        --no_labels \
        --batch_size 16


In [ ]:
# Cell 8: Summary
import os
import torch

os.chdir("/content/TrumpetJudge")

print("=" * 60)
print("📊 PRECOMPUTATION SUMMARY")
print("=" * 60)

files = [
    ("data/embeddings/all_audio.pt", "Original audio"),
    ("data/embeddings/all_augmented.pt", "Augmented audio (optional)"),
]

total_size = 0
for path, name in files:
    if os.path.exists(path):
        data = torch.load(path, weights_only=False)
        size_mb = os.path.getsize(path) / 1e6
        total_size += size_mb
        print(f"\n✅ {name}:")
        print(f"   File: {path}")
        print(f"   Samples: {data['num_samples']}")
        print(f"   Size: {size_mb:.1f} MB")
    else:
        print(f"\n⬜ {name}: Not created")

print(f"\n📦 Total size: {total_size:.1f} MB")
print("\n" + "=" * 60)
print("NEXT STEPS:")
print("=" * 60)
print("1. Download embeddings (next cell)")
print("2. Place in data/embeddings/ locally")
print("3. Run: python ml/prepare_data.py --labels data/labels/*.csv")
print("4. Train:")
print("   python ml/train_fast.py \\")
print("       --embeddings data/embeddings/all_audio.pt \\")
print("       --train_csv data/prepared/train.csv \\")
print("       --val_csv data/prepared/val.csv")


In [ ]:
# Cell 9: Download embeddings
import os
import shutil
from google.colab import files

os.chdir("/content/TrumpetJudge")

# Create zip of embeddings
print("📦 Creating embeddings archive...")
if os.path.exists("embeddings.zip"):
    os.remove("embeddings.zip")
shutil.make_archive("embeddings", 'zip', "data/embeddings")

size_mb = os.path.getsize("embeddings.zip") / 1e6
print(f"📥 Downloading embeddings.zip ({size_mb:.1f} MB)...")
files.download("embeddings.zip")

print("\n✅ Done! Extract to data/embeddings/ in your local repo.")


In [ ]:
# Cell 10 (Alternative): Save to Google Drive
import os
import shutil

os.chdir("/content/TrumpetJudge")

# ============================================
# CONFIGURE: Where to save in Google Drive
# ============================================
DRIVE_SAVE_PATH = "/content/drive/MyDrive/TrumpetJudge/embeddings"
# ============================================

os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

print("📦 Copying embeddings to Google Drive...")

for f in ["all_audio.pt", "all_augmented.pt"]:
    src = f"data/embeddings/{f}"
    dst = f"{DRIVE_SAVE_PATH}/{f}"
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(src) / 1e6
        print(f"   ✅ Copied {f} ({size_mb:.1f} MB)")
    else:
        print(f"   ⬜ Skipped {f} (not found)")

print(f"\n✅ Embeddings saved to: {DRIVE_SAVE_PATH}")
print("Access these anytime from Google Drive!")
